# CS229 Simplified PS2 — Classification and Neural Networks

参考 CS229 logistic stability、imbalanced labels 与 simple-NN 作业题型。

这不是 Stanford 官方作业副本，而是用于自学的**简化原创版本**。题型参考公开的
CS229 problem sets；公式以 2026 Main Notes 为准，数据由本仓库确定性生成。

- [Spring 2026 课程主页](https://cs229.stanford.edu/index.html-spr26)
- [Spring 2026 公开视频](https://www.youtube.com/playlist?list=PLaqpC4kq8Gpw)
- [CS229 2026 Main Notes](https://cs229.stanford.edu/main_notes.pdf)
- [Stanford 官方公开的 Summer 2020 problem sets](https://cs229.stanford.edu/summer2020/)
- [公开可见的 Summer 2025 作业结构参考](https://github.com/MDzimah/Stanford-University-CS229-Machine-Learning)

**使用规则：** 先手推，再写代码。不要先看学生 solution。每道题完成后才把对应的
`RUN_*_CHECKS` 改成 `True`。检查不通过时，从 shape 和公式开始定位。

**怎么读题：** `m` 表示样本数，`d` 表示设计矩阵的列数（含截距列时也计入）。
`X` 的每一行是一条样本，`y[i]` 是对应目标；`(m,)` 是一维数组，和 `(m, 1)` 不同。
“实现”指填写函数中的 TODO；“运行检查”指执行已经给出的调用代码；“解释”写在答题框中。
函数定义格和检查格需要分别运行。检查通过只代表已检查的条件成立，不代替解释实验结果。

## Goal

对应 Spring 2026 Lecture 3–8。先在 logistic regression 的语境里正式引入 binary
classifier 和 cross-entropy，再研究类别不平衡，最后实现一个两层神经网络。

## Setup

`label` 为 0 或 1。正类样本较少，这是题目故意设置的实验条件。

加载函数已给每行加入截距：`X_train` 为 `(200, 3)`，每行为 `[1, x1, x2]`，
`y_train` 为 `(200,)`；验证集 100 行。不要再添加一列 1。
训练集用于拟合，验证集用于比较；所有模型都用概率阈值 0.5 生成类别。
P2 需要先运行 P1 检查格得到预测，P3 复用 P1–P2 函数；P4 使用另外给出的 XOR 数据。

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(229)

candidates = [Path("data"), Path("problem_sets/ps2_classification/data")]
DATA_DIR = next((path for path in candidates if path.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Run: python scripts/build_problem_sets.py")

print("Data directory:", DATA_DIR.resolve())


In [ ]:
def load_classification(filename):
    data = np.loadtxt(DATA_DIR / filename, delimiter=",", skiprows=1)
    X = data[:, :2]
    y = data[:, 2].astype(int)
    X = np.column_stack([np.ones(len(X)), X])
    return X, y

X_train, y_train = load_classification("train.csv")
X_valid, y_valid = load_classification("valid.csv")
print("train:", X_train.shape, y_train.shape, "positive rate:", y_train.mean())
print("valid:", X_valid.shape, y_valid.shape, "positive rate:", y_valid.mean())


## Problem 1 — Logistic regression from scratch

**课堂连接：** Lecture 3，binary classification 与 logistic regression。

$$p(y=1\mid x;\theta)=\sigma(\theta^\top x),\qquad
\sigma(z)=\frac{1}{1+e^{-z}}.$$

**符号与目标：** 每个样本的三个输入（包括截距项）与参数 `theta` 内积得到分数 `z`，
sigmoid 把分数转换成正类概率 `p`。`z=X @ theta` 和 `p` 都为 `(m,)`，
`theta` 为 `(3,)`。概率不是类别；最后用 `p >= 0.5` 判为 1，否则判为 0。

**先推导：** 令 $p_i=\sigma(\theta^\top x_i)$，从平均二元交叉熵

$$L(\theta)=-\frac1m\sum_{i=1}^m[y_i\log p_i+(1-y_i)\log(1-p_i)]$$

出发，对 `theta` 求梯度并标 shape。这里使用自然对数。

**再实现三个函数，接口约定如下：**

- `sigmoid(z)`：逐元素转换，返回与 `z` 同 shape 的值。需能处理正负极大数，
  如 `np.array([-1000., 0., 1000.])`；可按正负分支改写公式，避免直接计算过大的指数。
- `binary_cross_entropy(y, probability, sample_weight=None)`：返回一个数。
  在取对数前把概率裁剪到 `[1e-12, 1-1e-12]`。
- `fit_logistic_regression(...)`：从零参数开始，每步使用全部训练样本的损失梯度更新，
  每次更新前保存一个 loss。返回最终 `(d,)` 参数和长度为 `steps` 的 `history`。

**为 P3 预留的权重规则：** `sample_weight=None` 表示每条样本权重为 1。
若传入 `(m,)` 的非负权重 $w_i$（总和大于 0），损失定义为

$$L_w=-\frac{\sum_i w_i[y_i\log p_i+(1-y_i)\log(1-p_i)]}{\sum_i w_i}.$$

梯度必须与这个加权损失一致，分母同样是权重总和；不能只在 loss 中加权。
先验证全 1 权重与 `None` 等价，再做 P3 的非均匀权重实验。

**交付与检查：** 推导写入答题框；运行下格，查看参数、前五个验证概率和下降的 loss。
此处的检查还不能判断少数类识别好不好，那是 P2 的任务。

> **你的答案（请双击本格后填写）**
>
> TODO：推导 binary cross-entropy gradient，并写出每个量的 shape。

In [ ]:
def sigmoid(z):
    # START TODO
    raise NotImplementedError("Implement a numerically stable sigmoid")
    # END TODO

def binary_cross_entropy(y, probability, sample_weight=None):
    # START TODO
    raise NotImplementedError("Clip probabilities and return weighted mean loss")
    # END TODO

def fit_logistic_regression(X, y, sample_weight=None, learning_rate=0.1, steps=2000):
    theta = np.zeros(X.shape[1])
    history = []
    for step in range(steps):
        # START TODO
        raise NotImplementedError("Compute probabilities, gradient, and update")
        # END TODO
    return theta, np.array(history)


In [ ]:
RUN_P1_CHECKS = False
if RUN_P1_CHECKS:
    extreme = sigmoid(np.array([-1000., 0., 1000.]))
    assert np.all(np.isfinite(extreme))
    assert np.allclose(extreme, [0., 0.5, 1.])
    assert np.isfinite(binary_cross_entropy(np.array([1., 0.]), np.array([0., 1.])))
    theta_log, history = fit_logistic_regression(X_train, y_train)
    probability = sigmoid(X_valid @ theta_log)
    prediction = (probability >= 0.5).astype(int)
    print("theta:", theta_log)
    print("first probabilities:", probability[:5])
    assert theta_log.shape == (3,)
    assert np.all((probability >= 0) & (probability <= 1))
    assert history[-1] < history[0]
else:
    print("P1 checks are off.")


## Problem 2 — 为什么 accuracy 会骗人

**题型来源：** 新版公开作业中的 imbalanced classification 实验。

**问题：** 验证集多数样本的标签为 0。如果把所有样本都猜成 0，总体正确率可能很高，
但真正的正类一个也找不到。本题要把“整体正确”和“能识别正类”区分开。

**函数输入：** `classification_metrics(y_true, y_pred)` 接收两个同长度 `(m,)`
的 0/1 数组；`y_pred` 是类别，不是概率。约定 1 为正类、0 为负类。

**需要返回的字典键与含义：**

| 键 | 定义 |
|---|---|
| `TP` | 真实为 1，预测也为 1 的数量 |
| `TN` | 真实为 0，预测也为 0 的数量 |
| `FP` | 真实为 0，却预测为 1 的数量 |
| `FN` | 真实为 1，却预测为 0 的数量 |
| `accuracy` | `(TP+TN)/m` |
| `precision` | `TP/(TP+FP)`：预测为正的样本中有多少是真的正类 |
| `recall` | `TP/(TP+FN)`：真实正类中有多少被找到了 |
| `specificity` | `TN/(TN+FP)`：真实负类中有多少被判对 |
| `balanced_accuracy` | `(recall+specificity)/2`：两个类别的正确率各占一半 |
| `f1` | `2*precision*recall/(precision+recall)` |

本练习约定上述指标的分母为 0 时返回 `0.0`，避免全零预测导致 precision 报错。
两个类别的样本数由 `y_true` 决定，不由模型预测决定。

**操作与交付：** 完成函数后先运行小例子检查，再在同一验证集比较全零 baseline 和
P1 的 `prediction`。在答题框引用两组的 accuracy、recall 和 balanced accuracy，
解释为什么仅报告 accuracy 会遗漏少数类失败。

In [ ]:
def classification_metrics(y_true, y_pred):
    # Return a dictionary containing counts and requested metrics.
    # START TODO
    raise NotImplementedError("Compute confusion counts and metrics")
    # END TODO


In [ ]:
RUN_P2_CHECKS = False
if RUN_P2_CHECKS:
    example = classification_metrics(np.array([0, 0, 1, 1]), np.array([0, 1, 0, 1]))
    assert all(example[key] == 1 for key in ("TP", "TN", "FP", "FN"))
    assert np.isclose(example["balanced_accuracy"], 0.5)
    all_zero = np.zeros_like(y_valid)
    print("all-zero baseline:", classification_metrics(y_valid, all_zero))
    print("logistic model  :", classification_metrics(y_valid, prediction))
else:
    print("P2 checks are off. Complete and run P1 first.")


> **你的答案（请双击本格后填写）**
>
> TODO：引用你算出的数字，解释为什么 stakeholder 不能只看 accuracy。

## Problem 3 — 给少数类更高权重

**目标：** 让损失函数更重视训练集中较少的正类，观察模型判别结果如何改变。
复用 P1 的加权训练函数及 P2 的指标函数，不修改验证集的标签或类别比例。

**需要自己完成下格的实验：**

1. 仅从 `y_train` 统计负类个数 `n0` 和正类个数 `n1`。
2. 构造 `(m,)` 的 `sample_weight`：负类权重为 1，正类权重为 `n0/n1`。
   这样两类的总权重相等。权重控制的是训练损失，不是改变标签。
3. 调用 `fit_logistic_regression` 重新从零训练，传入权重，保持普通模型的学习率和迭代次数。
4. 在原来的 `X_valid` 上得到概率，再用同一个阈值 0.5 转成类别。
5. 用 `classification_metrics` 并排输出普通模型与加权模型的指标。

**比较口径：** 本数据的 majority（多数类）是 0，其类内正确率为 `specificity`；
minority（少数类）是 1，其类内正确率为 `recall`。同时报告 `balanced_accuracy` 和 `precision`。
少数类 recall 可能提高，多数类正确率或 precision 可能下降；按实际数字解释，不要求所有指标提高。
在下格实验代码中用注释记录比较结果，以及哪类错误因此变多或变少。

In [ ]:
RUN_P3_CHECKS = False
if RUN_P3_CHECKS:
    # START TODO
    raise NotImplementedError("Create weights, refit, and compare metrics")
    # END TODO
else:
    print("P3 checks are off.")


## Problem 4 — 两层神经网络处理 XOR

**课堂连接：** Lecture 7–8。线性决策边界无法解决 XOR，两层网络可以。

**数据与任务：** XOR（异或）表示两个输入不同则标签为 1，相同则为 0。
下格给出四种输入和标签。本题只要求网络拟合这四个点，不把训练准确率当作泛化结果。
`X_xor` 为 `(4, 2)`，没有截距列；网络的偏置由 `b1`、`b2` 单独提供。

**前向计算约定：** 网络为 `2 -> 4 -> 1`，分别指输入维数、隐藏单元数、输出单元数。

$$Z_1=XW_1+b_1,\quad H=\tanh(Z_1),\quad Z_2=HW_2+b_2,\quad P=\sigma(Z_2).$$

`W1` 为 `(2,4)`，`b1` 为 `(4,)`，`W2` 为 `(4,1)`，`b2` 为 `(1,)`。
偏置沿样本行广播。先在纸上补出 `Z1`、`H`、`Z2`、`P` 的 shape，再推导平均二元交叉熵的梯度。

**需要实现的函数：**

- `mlp_forward(X, params)` 返回 `(probability, cache)`。`probability` 固定为 `(m,1)`；
  `cache` 使用字典，保存 `H` 和 `probability`，供反向传播复用。
- `mlp_backward(X, y, params, cache)` 返回字典，键必须是 `W1`、`b1`、`W2`、`b2`，
  每个梯度的 shape 与对应参数完全相同。先把 `(m,)` 的 `y` 转为 `(m,1)` 再与概率相减，
  否则广播会错误地产生 `(m,m)`。损失按样本取平均，因此梯度也需要按样本数归一化。

**实现提示与交付：** 对 `tanh` 求导时可复用隐藏层值，导数为 `1-H**2`。
在函数旁用注释写出各中间量 shape 和反向链式求导步骤；训练循环和参数更新已经提供。
运行 `RUN_P4_CHECKS`，记录四个概率，检查阈值 0.5 下四个标签均正确。
若不收敛，先检查梯度平均、偏置求和轴和参数 shape。

In [ ]:
X_xor = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y_xor = np.array([0., 1., 1., 0.])

def init_mlp(rng):
    return {
        "W1": rng.normal(scale=0.5, size=(2, 4)),
        "b1": np.zeros(4),
        "W2": rng.normal(scale=0.5, size=(4, 1)),
        "b2": np.zeros(1),
    }

def mlp_forward(X, params):
    # START TODO
    raise NotImplementedError("Return probability and a cache for backprop")
    # END TODO

def mlp_backward(X, y, params, cache):
    # START TODO
    raise NotImplementedError("Return gradients matching every parameter shape")
    # END TODO


In [ ]:
RUN_P4_CHECKS = False
if RUN_P4_CHECKS:
    params = init_mlp(np.random.default_rng(229))
    for step in range(5000):
        probability, cache = mlp_forward(X_xor, params)
        grads = mlp_backward(X_xor, y_xor, params, cache)
        for name in params:
            assert grads[name].shape == params[name].shape
            params[name] -= 0.1 * grads[name]
    probability, _ = mlp_forward(X_xor, params)
    print("XOR probabilities:", probability.ravel())
    assert np.mean((probability.ravel() >= 0.5) == y_xor) == 1.0
else:
    print("P4 checks are off.")


## Checks

- [ ] 我能解释 classifier 为什么从 Lecture 3 才出现。
- [ ] sigmoid 和 cross-entropy 在极端输入下不产生 NaN。
- [ ] 我报告了 minority-class 指标，而不只报告 accuracy。
- [ ] backprop 返回的每个梯度与对应参数 shape 相同。

## Next Steps

完成后进入 PS3，处理没有标签或带隐变量的数据。